# ***Libraries, Tools & Definitions***

In [ ]:
import pandas as pd
import numpy as np
import nltk
import sklearn as sk
import re
import statistics
import os
import json
import random
import seaborn as sns
import string
import spacy
import matplotlib.pyplot as plt
import yake
import pytextrank
import time
import gc
#import sys; sys.path.append("..")
#import pke # Use this when you want to run the KE methods in pke

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering

from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


from kneed import KneeLocator
from itertools import chain, groupby, product
from enum import Enum
from typing import Callable, DefaultDict, Dict, List, Optional, Set, Tuple
from collections import Counter, defaultdict
from gensim.models import KeyedVectors
from tqdm.notebook import tqdm

In [2]:
nltk.data.path.append("/home/georgematlis/AUTh/NLP/Keyword-Extraction/Downloads") # Path to tokenizers, corpora and taggers

In [ ]:
parent_path = '../MDPI Articles/'

#target_folder = 'MAKE 1996-2024/'
target_folder = 'Applied Sci 1996-2024/'
target_folders = ["MAKE 1996-2024/", "Applied Sci 1996-2024/", "Algo 1996-2024/", "IJMS 1996-2024/", 'Sensors 1996-2024/', 'Sustainability 1996-2024/'] 


article_data_file = 'MDPI_Applied_Sci_Augmented_Content.json'
article_data_files = ['MDPI_MAKE_Augmented_Content.json', 
                      'MDPI_Applied_Sci_Augmented_Content.json',
                      'MDPI_Algo_Augmented_Content.json',
                      'MDPI_ijms_Augmented_Content.json',
                      'MDPI_Sensors_Augmented_Content.json',
                      'MDPI_sustainability_Augmented_Content.json']

spacy_package = '../en_core_web_sm-3.8.0-py3-none-any/en_core_web_sm/en_core_web_sm-3.8.0' # Used as pipeline for graph-based methods (TextRank, PositionRank, TopicRank)

In [4]:
stemmer = PorterStemmer()

In [5]:
# Tokenize and stem keywords
def stem_tokens(kw):
    return [stemmer.stem(w.lower()) for w in word_tokenize(kw)]

In [ ]:
# Similarity function: overlap / max(lenA, lenB)
def keyword_similarity(set_a, set_b): # Based on exact similarity 
    overlap = len(set(set_a) & set(set_b))
    denom = max(len(set_a), len(set_b))
    return overlap / denom if denom > 0 else 0.0

# ***Semantic Similarity***

## SpaCy

In [ ]:
#spaCy = spacy.load("../en_core_web_lg-3.7.1-py3-none-any/en_core_web_lg/en_core_web_lg-3.7.1")
word_embedding_model = spacy.load("../en_core_web_lg-3.8.0-py3-none-any/en_core_web_lg/en_core_web_lg-3.8.0")

## Word2Vec Model with Pre-Trained Embeddings

In [7]:
# Load the Google News embeddings
word_embedding_model = KeyedVectors.load_word2vec_format('../GoogleNews-vectors-negative300.bin', binary=True)

## Get Embeddings for Keywords  

In [ ]:
# Create embeddings for each keyword by averaging word embeddings
def get_keyword_embedding(keyword, default_vector=None):
    words = keyword.split()
    valid_vectors = []
    
    for word in words:
        if word in word_embedding_model:
            valid_vectors.append(word_embedding_model[word])
        elif default_vector is not None:
            valid_vectors.append(default_vector)
    
    if valid_vectors:
        return np.mean(valid_vectors, axis=0) # Average the word embeddings of a keyword
    
    return None

# ***MDPI Data Load & Definitions***

## Load One JSON Data File

In [ ]:
with open(f'{parent_path}{target_folder}{article_data_file}', 'r') as file:
    data = json.load(file)

titles = data["Titles"]
articles = data["Articles"]
keywords_list = data["Keywords"]

print("Titles:", titles)
print("Articles:", articles)
print("Keywords:", keywords_list)

Titles: ['Embrace the Era of Drones: A New Practical Design Approach to Emergency Rescue Drones', 'Automated Dead Chicken Detection in Poultry Farms Using Knowledge Distillation and Vision Transformers', 'Reuse of Spent Coffee Grounds: Alternative Applications, Challenges, and Prospects—A Review', 'Exploiting Extrinsic Information for Serial MAP Detection by Utilizing Estimator in Holographic Data Storage Systems', 'A Novel Optimization Method Using the Box–Behnken Design Integrated with a Back Propagation Neural Network–Genetic Algorithm for Hydrogen Purification', 'Analysis of Wind Speed Characteristics Along a High-Speed Railway', 'Imputing Covariance for Meta-Analysis in the Presence of Interaction', 'Energy Management Strategies for Extended-Range Electric Vehicles with Real Driving Emission Constraints', 'Content of Selected Macroelements and Zinc in Relation to Stage of Lactation of Montbéliarde and Polish Holstein-Friesian Cows', 'Circular Economy Alternative Repurposing Textil

In [ ]:
with open(f'{parent_path}{target_folder}{article_data_file}', 'r') as file:
    data = json.load(file)

titles = data["Titles"]
articles = data["Abstracts"]
keywords_list = data["Keywords"]

print("Titles:", titles)
print("Abstracts:", articles)
print("Keywords:", keywords_list)

In [8]:
print(len(titles))
print(len(articles))
print(len(keywords_list))

100
100
100


In [14]:
# Compute embeddings for each article's keywords
all_keyword_embeddings = [
    [get_keyword_embedding(kw) for kw in keywords]
    for keywords in keywords_list
]

In [9]:
#keyword_comparison_sim = 0.9
T = 5
#T = [5, 10]

## Multiple JSON Data Files

In [ ]:
def load_multiple_jsons(parent_path, target_folders, article_data_files):
    all_titles, all_articles, all_keywords = [], [], []

    data_file_index = 0
    for folder in target_folders:
        file_path = os.path.join(parent_path, folder, article_data_files[data_file_index])
        
        with open(file_path, "r") as file: # encoding='utf-8'
            data = json.load(file)

        all_titles.extend(data.get("Titles", []))
        all_articles.extend(data.get("Articles", [])) # You might want to change this in data.get("Abstracts", []) when using only abstracts
        all_keywords.extend(data.get("Keywords", []))

        data_file_index += 1

    return all_titles, all_articles, all_keywords

In [10]:
titles, articles, keywords_list = load_multiple_jsons(parent_path, target_folders, article_data_files)

In [11]:
print("Titles:",   len(titles))
print("Articles:", len(articles))
print("Keywords:", len(keywords_list))

Titles: 1179
Articles: 1179
Keywords: 1179


## Keyword Statistics

In [ ]:
keywords_list

In [12]:
print("Median number of keywords:", statistics.median([len(sublist) for sublist in keywords_list]))

Median number of keywords: 5


# ***Text Extraction***

## RAKE Implementation from https://csurfer.github.io/rake-nltk/_build/html/_modules/rake_nltk/rake.html

In [15]:
# Readability type definitions.
Word = str
Sentence = str
Phrase = Tuple[str, ...]

In [16]:
class Metric(Enum):
    """Different metrics that can be used for ranking."""

    DEGREE_TO_FREQUENCY_RATIO = 0  # Uses d(w)/f(w) as the metric
    WORD_DEGREE = 1  # Uses d(w) alone as the metric
    WORD_FREQUENCY = 2  # Uses f(w) alone as the metric

In [17]:
class Rake:
    """Rapid Automatic Keyword Extraction Algorithm."""

    def __init__(
        self,
        stopwords: Optional[Set[str]] = None,
        punctuations: Optional[Set[str]] = None,
        language: str = 'english',
        ranking_metric: Metric = Metric.DEGREE_TO_FREQUENCY_RATIO,
        max_length: int = 100000,
        min_length: int = 1,
        include_repeated_phrases: bool = True,
        sentence_tokenizer: Optional[Callable[[str], List[str]]] = None,
        word_tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        """Constructor.

        :param stopwords: Words to be ignored for keyword extraction.
        :param punctuations: Punctuations to be ignored for keyword extraction.
        :param language: Language to be used for stopwords.
        :param max_length: Maximum limit on the number of words in a phrase
                           (Inclusive. Defaults to 100000)
        :param min_length: Minimum limit on the number of words in a phrase
                           (Inclusive. Defaults to 1)
        :param include_repeated_phrases: If phrases repeat in phrase list consider
                            them as is without dropping any phrases for future
                            calculations. (Defaults to True) Ex: "Magic systems is
                            a company. Magic systems was founded by Raul".

                            If repeated phrases are allowed phrase list would be
                            [
                                (magic, systems), (company,), (magic, systems),
                                (founded,), (raul,)
                            ]

                            If they aren't allowed phrase list would be
                            [
                                (magic, systems), (company,),
                                (founded,), (raul,)
                            ]
        :param sentence_tokenizer: Tokenizer used to tokenize the text string into sentences.
        :param word_tokenizer: Tokenizer used to tokenize the sentence string into words.
        """
        # By default use degree to frequency ratio as the metric.
        if isinstance(ranking_metric, Metric):
            self.metric = ranking_metric
        else:
            self.metric = Metric.DEGREE_TO_FREQUENCY_RATIO

        # If stopwords not provided we use language stopwords by default.
        self.stopwords: Set[str]
        if stopwords:
            self.stopwords = stopwords
        else:
            self.stopwords = set(nltk.corpus.stopwords.words(language))

        # If punctuations are not provided we ignore all punctuation symbols.
        self.punctuations: Set[str]
        if punctuations:
            self.punctuations = punctuations
        else:
            self.punctuations = set(string.punctuation)

        # All things which act as sentence breaks during keyword extraction.
        self.to_ignore: Set[str] = set(chain(self.stopwords, self.punctuations))

        # Assign min or max length to the attributes
        self.min_length: int = min_length
        self.max_length: int = max_length

        # Whether we should include repeated phreases in the computation or not.
        self.include_repeated_phrases: bool = include_repeated_phrases

        # Tokenizers.
        self.sentence_tokenizer: Callable[[str], List[str]]
        if sentence_tokenizer:
            self.sentence_tokenizer = sentence_tokenizer
        else:
            self.sentence_tokenizer = nltk.tokenize.sent_tokenize
        self.word_tokenizer: Callable[[str], List[str]]
        if word_tokenizer:
            self.word_tokenizer = word_tokenizer
        else:
            self.word_tokenizer = nltk.tokenize.wordpunct_tokenize

        # Stuff to be extracted from the provided text.
        self.frequency_dist: Dict[Word, int]
        self.degree: Dict[Word, int]
        self.rank_list: List[Tuple[float, Sentence]]
        self.ranked_phrases: List[Sentence]

    def extract_keywords_from_text(self, text: str):
        """Method to extract keywords from the text provided.

        :param text: Text to extract keywords from, provided as a string.
        """
        sentences: List[Sentence] = self._tokenize_text_to_sentences(text)
        self.extract_keywords_from_sentences(sentences)

    def extract_keywords_from_sentences(self, sentences: List[Sentence]):
        """Method to extract keywords from the list of sentences provided.

        :param sentences: Text to extraxt keywords from, provided as a list
                          of strings, where each string is a sentence.
        """
        phrase_list: List[Phrase] = self._generate_phrases(sentences)
        self._build_frequency_dist(phrase_list)
        self._build_word_co_occurance_graph(phrase_list)
        self._build_ranklist(phrase_list)

    def get_ranked_phrases(self) -> List[Sentence]:
        """Method to fetch ranked keyword strings.

        :return: List of strings where each string represents an extracted
                 keyword string.
        """
        return self.ranked_phrases

    def get_ranked_phrases_with_scores(self) -> List[Tuple[float, Sentence]]:
        """Method to fetch ranked keyword strings along with their scores.

        :return: List of tuples where each tuple is formed of an extracted
                 keyword string and its score. Ex: (5.68, 'Four Scoures')
        """
        return self.rank_list

    def get_word_frequency_distribution(self) -> Dict[Word, int]:
        """Method to fetch the word frequency distribution in the given text.

        :return: Dictionary (defaultdict) of the format `word -> frequency`.
        """
        return self.frequency_dist

    def get_word_degrees(self) -> Dict[Word, int]:
        """Method to fetch the degree of words in the given text. Degree can be
        defined as sum of co-occurances of the word with other words in the
        given text.

        :return: Dictionary (defaultdict) of the format `word -> degree`.
        """
        return self.degree

    def _tokenize_text_to_sentences(self, text: str) -> List[Sentence]:
        """Tokenizes the given text string into sentences using the configured
        sentence tokenizer. Configuration uses `nltk.tokenize.sent_tokenize`
        by default.

        :param text: String text to tokenize into sentences.
        :return: List of sentences as per the tokenizer used.
        """
        return self.sentence_tokenizer(text)

    def _tokenize_sentence_to_words(self, sentence: Sentence) -> List[Word]:
        """Tokenizes the given sentence string into words using the configured
        word tokenizer. Configuration uses `nltk.tokenize.wordpunct_tokenize`
        by default.

        :param sentence: String sentence to tokenize into words.
        :return: List of words as per the tokenizer used.
        """
        return self.word_tokenizer(sentence)

    def _build_frequency_dist(self, phrase_list: List[Phrase]) -> None:
        """Builds frequency distribution of the words in the given body of text.

        :param phrase_list: List of List of strings where each sublist is a
                            collection of words which form a contender phrase.
        """
        self.frequency_dist = Counter(chain.from_iterable(phrase_list))

    def _build_word_co_occurance_graph(self, phrase_list: List[Phrase]) -> None:
        """Builds the co-occurance graph of words in the given body of text to
        compute degree of each word.

        :param phrase_list: List of List of strings where each sublist is a
                            collection of words which form a contender phrase.
        """
        co_occurance_graph: DefaultDict[Word, DefaultDict[Word, int]] = defaultdict(lambda: defaultdict(lambda: 0))
        for phrase in phrase_list:
            # For each phrase in the phrase list, count co-occurances of the
            # word with other words in the phrase.
            #
            # Note: Keep the co-occurances graph as is, to help facilitate its
            # use in other creative ways if required later.
            for (word, coword) in product(phrase, phrase):
                co_occurance_graph[word][coword] += 1
        self.degree = defaultdict(lambda: 0)
        for key in co_occurance_graph:
            self.degree[key] = sum(co_occurance_graph[key].values())

    def _build_ranklist(self, phrase_list: List[Phrase]):
        """Method to rank each contender phrase using the formula

              phrase_score = sum of scores of words in the phrase.
              word_score = d(w) or f(w) or d(w)/f(w) where d is degree
                           and f is frequency.

        :param phrase_list: List of List of strings where each sublist is a
                            collection of words which form a contender phrase.
        """
        self.rank_list = []
        for phrase in phrase_list:
            rank = 0.0
            for word in phrase:
                if self.metric == Metric.DEGREE_TO_FREQUENCY_RATIO:
                    rank += 1.0 * self.degree[word] / self.frequency_dist[word]
                elif self.metric == Metric.WORD_DEGREE:
                    rank += 1.0 * self.degree[word]
                else:
                    rank += 1.0 * self.frequency_dist[word]
            self.rank_list.append((rank, ' '.join(phrase)))
        self.rank_list.sort(reverse=True)
        self.ranked_phrases = [ph[1] for ph in self.rank_list]

    def _generate_phrases(self, sentences: List[Sentence]) -> List[Phrase]:
        """Method to generate contender phrases given the sentences of the text
        document.

        :param sentences: List of strings where each string represents a
                          sentence which forms the text.
        :return: Set of string tuples where each tuple is a collection
                 of words forming a contender phrase.
        """
        phrase_list: List[Phrase] = []
        # Create contender phrases from sentences.
        for sentence in sentences:
            word_list: List[Word] = [word.lower() for word in self._tokenize_sentence_to_words(sentence)]
            phrase_list.extend(self._get_phrase_list_from_words(word_list))

        # Based on user's choice to include or not include repeated phrases
        # we compute the phrase list and return it. If not including repeated
        # phrases, we only include the first occurance of the phrase and drop
        # the rest.
        if not self.include_repeated_phrases:
            unique_phrase_tracker: Set[Phrase] = set()
            non_repeated_phrase_list: List[Phrase] = []
            for phrase in phrase_list:
                if phrase not in unique_phrase_tracker:
                    unique_phrase_tracker.add(phrase)
                    non_repeated_phrase_list.append(phrase)
            return non_repeated_phrase_list

        return phrase_list

    def _get_phrase_list_from_words(self, word_list: List[Word]) -> List[Phrase]:
        """Method to create contender phrases from the list of words that form
        a sentence by dropping stopwords and punctuations and grouping the left
        words into phrases. Only phrases in the given length range (both limits
        inclusive) would be considered to build co-occurrence matrix. Ex:

        Sentence: Red apples, are good in flavour.
        List of words: ['red', 'apples', ",", 'are', 'good', 'in', 'flavour']
        List after dropping punctuations and stopwords.
        List of words: ['red', 'apples', *, *, good, *, 'flavour']
        List of phrases: [('red', 'apples'), ('good',), ('flavour',)]

        List of phrases with a correct length:
        For the range [1, 2]: [('red', 'apples'), ('good',), ('flavour',)]
        For the range [1, 1]: [('good',), ('flavour',)]
        For the range [2, 2]: [('red', 'apples')]

        :param word_list: List of words which form a sentence when joined in
                          the same order.
        :return: List of contender phrases honouring phrase length requirements
                 that are formed after dropping stopwords and punctuations.
        """
        groups = groupby(word_list, lambda x: x not in self.to_ignore)
        phrases: List[Phrase] = [tuple(group[1]) for group in groups if group[0]]
        return list(filter(lambda x: self.min_length <= len(x) <= self.max_length, phrases))

## TF-IDF Function

In [ ]:
def extract_keywords_tfidf(preprocessed_content, T):
    # Initialize TF-IDF vectorizer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(preprocessed_content)
    
    # Get feature names (i.e., words)
    feature_names = vectorizer.get_feature_names_out()
    
    # Convert the TF-IDF matrix to a DataFrame
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)
    
    # Extract T keywords for each article
    keywords_per_article = []
    for _, row in tfidf_df.iterrows():
        # Get the T words with highest TF-IDF scores
        top_keywords = row.nlargest(T).index.tolist()
        keywords_per_article.append(top_keywords)
    
    return keywords_per_article

## ***Evaluation of Text Extraction Methods***

### Helper Functions

In [19]:
def sample_keywords(keywords, sampling_percentage):
    return keywords[:int(len(keywords) * (sampling_percentage / 100.0))]

def is_valid_word(word):
    # Check if the word contains only letters
    return re.match(r'^[A-Za-z]+$', word) is not None


# Used for exact comparrissons
def compare_keyword_lists(list1, list2):
    kws1 = [k.lower() for k in list1]
    kws2 = [k.lower() for k in list2]

    set1, set2 = set(kws1), set(kws2)
    return len(set1 & set2)#, len(set1 ^ set2)

#### Evaluation Functions

In [ ]:
# This function calculates how many words and non-words (numbers and symbols) exist in the keywords list
def process_keywords(keywords_list, output_file=None):
    word_counts = []  # To store word count per keyword list
    total_word_count = 0
    total_non_word_count = 0

    for keywords in keywords_list:
        keywords_word_count = 0  # Count words in this keywords list
        keywords_non_word_count = 0

        for keyword in keywords:
            # Split on whitespace or dash but preserve valid words
            words = [word for part in keyword.split('-') for word in part.split()]
            
            for word in words:
                if word.isalpha():  # Check if it's a valid word
                    keywords_word_count += 1
                    total_word_count += 1
                else:
                    keywords_non_word_count += 1
                    total_non_word_count += 1

        word_counts.append(keywords_word_count)
    
    avg_word_count = 0
    for count in word_counts:
        avg_word_count += count

    if output_file:
        with open(output_file, 'w') as f:
            for count in word_counts:
                f.write(str(count) + '\n')
    
    return total_word_count, total_non_word_count, avg_word_count / len(word_counts)

In [ ]:
# This is used for partial matches
def count_word_overlap_matches(candidate_keywords, reference_keywords, threshold=0.25):
    """
    Count how many candidate keywords match the reference keywords
    based on word overlap (≥ threshold).
    
    A match means: at least `threshold` fraction of words in a candidate
    keyword appear in the words of SOME reference keyword.
    """
    matches = 0
    matched_indices = set()  # To avoid matching the same reference keyword multiple times

    for cand_kw in candidate_keywords:
        cand_words = cand_kw.lower().split()
        cand_len = len(cand_words)
        if cand_len == 0:
            continue

        for idx, ref_kw in enumerate(reference_keywords):
            if idx in matched_indices:
                continue
            
            ref_words = set(ref_kw.lower().split()) # We use a set for fast lookups
            overlap = sum(1 for w in cand_words if w in ref_words)

            if overlap / cand_len >= threshold:
                matches += 1
                matched_indices.add(idx)
                break  # Stop once we match this candidate to one reference keyword
    
    return matches

In [ ]:
# This function is used when we evaluate any KE method
def evaluate_method(name, true_positives, extracted_keywords, true_keywords):
    precision = true_positives / extracted_keywords if extracted_keywords > 0 else 0
    recall = true_positives / true_keywords if true_keywords > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f'=== {name} ===')
    print(f'Precision: {precision}\nRecall: {recall}\nF1-Score: {f1}\n')
    print(f'Number of extracted keywords: {extracted_keywords}')
    print(f'Number of true keywords: {true_keywords}')
    print(f'True positives: {true_positives}\n')

In [ ]:
# This function is used when we evaluate one specific KE method
def evaluate_method(true_positives, extracted_keywords, true_keywords):
    precision = true_positives / extracted_keywords if extracted_keywords > 0 else 0
    recall = true_positives / true_keywords if true_keywords > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1

In [ ]:
def analyze_word_counts(filenames):
    # Store word counts for each file
    word_counts_per_file = []
    means = []

    # Read word counts from each file
    for filename in filenames:
        with open(filename, 'r') as file:
            word_counts = [int(line.strip()) for line in file]  # Convert each line to an integer
            word_counts_per_file.append(word_counts)
            means.append(sum(word_counts) / len(word_counts) if word_counts else 0)

    # Print the means
    for filename, mean in zip(filenames, means):
        print(f"Mean word count for {filename}: {mean}")

    labels = ['TF-IDF', 'RAKE', 'YAKE', 'TextRank', 'PositionRank','TopicRank']

    # Customize flier (outlier) properties
    flierprops = dict(marker='o', markersize=4, linestyle='none', markeredgecolor='black')

    # Create horizontal boxplots
    plt.figure(figsize=(12, 8))  # Increase figure size
    colors = sns.color_palette("pastel", len(filenames))  # Generate distinct colors
    boxplots = plt.boxplot(word_counts_per_file, patch_artist=True, widths=0.6, flierprops=flierprops, vert=False)

    # Apply colors to the boxes
    for patch, color in zip(boxplots['boxes'], colors):
        patch.set_facecolor(color)

    # Add gridlines
    plt.grid(axis='x', linestyle='--', alpha=0.7)

    # Customize y-axis labels (since plot is now horizontal)
    plt.yticks(
        ticks=range(1, len(filenames) + 1), 
        labels=labels,  # Simplified names
        fontsize=20
    )

    # Add title and labels
    plt.title("Word Count Distribution per Method", fontsize=20)
    plt.xlabel("Word Count", fontsize=20)
    plt.ylabel("Methods", fontsize=20)
    plt.xticks(fontsize=20)
    plt.tight_layout()

    # Save plots
    plt.savefig(f'{parent_path}Box_Plots_K5.svg', format="svg")
    plt.savefig(f'{parent_path}Box_Plots_K5.png', format="png")
    plt.savefig(f'{parent_path}Box_Plots_K5.jpg', format="jpg")

    # Show plot
    plt.show()


In [ ]:
# This function calculates the embedding similarity (cosine) of the candidates keywords and the true keywords of ONE article 
def count_matching_keywords(candidate_keywords, article_index, sim_threshold=0.8):
    """Count how many candidate keywords match the true keywords for an article."""
    
    # Embed all candidate keywords
    candidate_embeddings = [get_keyword_embedding(kw) for kw in candidate_keywords]
    reference_embeddings = all_keyword_embeddings[article_index]
    
    matched_indices = set()
    matches = 0
    
    for cand_emb in candidate_embeddings:
        for idx, ref_emb in enumerate(reference_embeddings):
            if idx in matched_indices:
                continue
            try:
                similarity = cosine_similarity([cand_emb], [ref_emb])[0][0]
                if similarity >= sim_threshold:
                    matches += 1
                    matched_indices.add(idx)
                    break  # stop searching after first match
            except ValueError:
                continue
    
    return matches

#### Knee Method

In [30]:
def find_knee_with_kneedle(scores):
    """
    Find the knee (elbow) in a list of sorted scores (descending) using Kneedle.
    
    Args:
        scores (list or np.array): Sorted list of scores (high to low).
    
    Returns:
        list: Indices of keywords to keep (up to and including knee).
    """
    scores = np.array(scores)
    x = np.arange(len(scores))

    # Kneedle works best if curve is convex and decreasing
    kneedle = KneeLocator(x, scores, curve="convex", direction="decreasing")
    knee = kneedle.knee

    if knee is None:
        # If no knee detected, keep all
        return list(range(len(scores)))
    else:
        return list(range(knee + 1))

#### HAC

In [ ]:
def cluster_keywords(keywords, scores, similarity_threshold=0.25):
    """
    Cluster keywords using Hierarchical Agglomerative Clustering (average linkage)
    with word overlap similarity, and return centroid keywords with their scores.

    Args:
        keywords (list of str): Candidate keywords.
        scores (list of float): Scores corresponding to each keyword.
        similarity_threshold (float): Minimum overlap similarity (default=0.25).

    Returns:
        list of (keyword, score): Cluster centroids and their scores.
    """

    stemmed_keywords = [stem_tokens(kw) for kw in keywords]

    # Build similarity matrix
    n = len(keywords)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            sim = keyword_similarity(stemmed_keywords[i], stemmed_keywords[j])
            sim_matrix[i, j] = sim
            sim_matrix[j, i] = sim

    # Convert to distance matrix
    dist_matrix = 1 - sim_matrix

    # Clustering with average linkage
    clustering = AgglomerativeClustering(
        metric="precomputed",
        linkage="average",
        distance_threshold=1 - similarity_threshold,
        compute_full_tree=True,
        n_clusters=None # There is no need to define the number of clusters
    )
    labels = clustering.fit_predict(dist_matrix)

    # Find cluster centroids
    centroids = []
    for cluster_id in set(labels): # Take all the unique labels, e.g., 0, 1, 2, 3, ...
        indices = [idx for idx, lbl in enumerate(labels) if lbl == cluster_id] # Organize the keyword indices based on their label
        if len(indices) == 1:
            # Single keyword cluster
            idx = indices[0]
            centroids.append((keywords[idx], scores[idx]))
        else:
            # Compute centroid: max avg similarity within cluster
            best_idx, best_sim = None, -1
            for idx in indices:
                sims = [sim_matrix[idx, j] for j in indices if j != idx]
                avg_sim = np.mean(sims) if sims else 0
                if avg_sim > best_sim:
                    best_sim = avg_sim
                    best_idx = idx
            centroids.append((keywords[best_idx], scores[best_idx]))

    return centroids


#### LLM Input Generation

In [ ]:
def generate_LLM_input(extracted_keywords):
    LLM_input = []

    for i, ekws in enumerate(extracted_keywords):
        # Separate scores and keywords
        scores, keywords = zip(*[(score, keyword) for score, keyword in ekws])

        # Cluster and sort by score descending
        centroids = sorted(cluster_keywords(list(keywords), list(scores), similarity_threshold=0.25), key=lambda x: x[1], reverse=True)

        # Apply knee method on scores
        knee_indices = find_knee_with_kneedle([score for _, score in centroids])


        # Given the length of each article's text, which is large, there will always be more than enough centroid keywords to use
        if len(knee_indices) < T:
            if T == 5:
                # Fill up to exactly 5
                extra = list(range(len(knee_indices), min(len(centroids), 5)))
                knee_indices.extend(extra)
            elif T == 10:
                # Fill up to at least 8 (if available)
                extra = list(range(len(knee_indices), min(len(centroids), 8)))
                knee_indices.extend(extra)

        # Build best centroids string
        top_centroids = " ".join(
            f"({round(centroids[idx][1], 2)}){centroids[idx][0]}"
            for idx in knee_indices
        )

        # Create LLM input string
        LLM_input.append(f"TITLE: {titles[i]}; KEYWORDS: {top_centroids}")
    
    return LLM_input


#### Generate Top Centroid Keywords 

In [29]:
def get_top_centroids(extracted_keywords):
    centroid_keywords = []

    for i, ekws in enumerate(extracted_keywords):
        # Separate scores and keywords
        scores, keywords = zip(*[(score, keyword) for score, keyword in ekws])

        # Cluster and sort by score descending
        centroids = sorted(cluster_keywords(list(keywords), list(scores), similarity_threshold=0.25), key=lambda x: x[1], reverse=True)

        # Apply knee method on scores
        knee_indices = find_knee_with_kneedle([score for _, score in centroids])

        if len(knee_indices) < T:
            if T == 5:
                # Fill up to exactly 5
                extra = list(range(len(knee_indices), min(len(centroids), 5)))
                knee_indices.extend(extra)
            elif T == 10:
                # Fill up to at least 8 (if available)
                extra = list(range(len(knee_indices), min(len(centroids), 8)))
                knee_indices.extend(extra)

        # Build best centroids string
        centroid_keywords.append([centroids[idx][0] for idx in knee_indices])
    
    return centroid_keywords

### TF-IDF Extraction

In [ ]:
# Preprocess the articles: remove punctuation and stopwords
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [21]:
def preprocess(text):
  text = text.lower()  # Lowercase
  text = ''.join([ch for ch in text if ch not in punctuation])  # Remove punctuation
  tokens = text.split()
  tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
  return ' '.join(tokens)

In [22]:
preprocessed_content = [preprocess(content) for content in articles]

In [ ]:
start_time = time.perf_counter()
tfidf_keywords = extract_keywords_tfidf(preprocessed_content, T)
print(f'TF-IDF time: {(time.perf_counter() - start_time) / 60.0}')

In [ ]:
total_word_count, total_non_word_count = process_keywords(tfidf_keywords)
print(total_non_word_count / total_word_count)

### ***Other Text Extraction Methods***

#### ***Single Execution***
No statistics

In [ ]:
#topicrank = spacy.load("en_core_web_sm-3.8.0-py3-none-any/en_core_web_sm/en_core_web_sm-3.8.0")
positionrank = spacy.load("../en_core_web_sm-3.8.0-py3-none-any/en_core_web_sm/en_core_web_sm-3.8.0")
#textrank = spacy.load("en_core_web_sm-3.8.0-py3-none-any/en_core_web_sm/en_core_web_sm-3.8.0")

In [ ]:
#textrank.add_pipe("textrank")
positionrank.add_pipe("positionrank")
#topicrank.add_pipe("topicrank")

In [9]:
lda_model_file = r'../pke/models/lda-1000-semeval2010.py3.pickle.gz'
weights_file = r'../pke/models/df-semeval2010.tsv.gz'
stoplist = list(stop_words) + list(punctuation)

In [ ]:
#yake_extractor = yake.KeywordExtractor(lan='en', n=3, dedupLim=0.9, dedupFunc='seqm', windowsSize=1, top=T, features=None)

final_extr_keyword_num = 0
true_keyword_num = 0
extracted_keywords = []

rake_tp      = 0
yake_tp      = 0
tfidf_tp     = 0
textr_tp     = 0
posr_tp      = 0
topicr_tp    = 0
singler_tp   = 0
kpm_tp       = 0
topicalpr_tp = 0
mprank_tp    = 0

#article_limit = int(len(articles) * (100.0 / 100.0))
#article_limit = 25

for i in range(len(articles)):

    #if i == article_limit: break

    current_article = titles[i] + '. ' + articles[i]
    true_keyword_num += len(keywords_list[i])
    final_extr_keyword_num += T

    # ===== SingleRank =====
    """ singlerank = pke.unsupervised.SingleRank()
    singlerank.load_document(input = current_article, language = 'en', normalization = None, spacy_model = spaCy)
    singlerank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
    singlerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'})
    extracted_keywords.append([key for key, _ in singlerank.get_all_candidates()])
    #singler_tp += count_word_overlap_matches([key for key, _ in singlerank.get_n_best(n=T)], keywords_list[i]) """
    
    """ # ===== TopicalPageRank =====
    topicalpagerank = pke.unsupervised.TopicalPageRank()
    topicalpagerank.load_document(input = current_article, language = 'en', normalization = None, spacy_model = spaCy)
    topicalpagerank.candidate_selection(grammar = "NP: {<ADJ>*<NOUN|PROPN>+}")
    topicalpagerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'}, lda_model = lda_model_file)
    extracted_keywords.append([key for key, _ in topicalpagerank.get_all_candidates()])
    #topicalpr_tp += count_word_overlap_matches([key for key,_ in topicalpagerank.get_n_best(n=T)], keywords_list[i]) """

    """ # ===== MPRank =====
    mprank = pke.unsupervised.MultipartiteRank()
    mprank.load_document(input = current_article, stoplist = stoplist, language = 'en', spacy_model=spaCy)
    mprank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
    mprank.candidate_weighting(alpha = 1.1, threshold = 0.74, method = 'average')
    extracted_keywords.append([key for key, _ in mprank.get_all_candidates()])
    #mprank_tp += count_word_overlap_matches([key for key,_ in mprank.get_n_best(n=T)], keywords_list[i]) """

    
    """ # ===== RAKE =====
    rake = Rake(ranking_metric=Metric.DEGREE_TO_FREQUENCY_RATIO)
    rake.extract_keywords_from_text(current_article)
    rake_keywords = sorted(set(rake.get_ranked_phrases_with_scores()), key=lambda x: x[0], reverse=True); extracted_keywords.append(rake_keywords)
    #rake_keywords = sample_keywords(rake_keywords, 100.0)[:T] # Use this if you don't want all the keywords
    #rake_tp += count_matching_keywords([kw for _, kw in rake_keywords[:T]], i)
    rake_tp += count_word_overlap_matches([kw for _, kw in rake_keywords[:T]], keywords_list[i])
    
    # ===== YAKE =====
    yake_keywords = yake_extractor.extract_keywords(current_article); extracted_keywords.append(yake_keywords)
    yake_tp += count_matching_keywords([kw for kw, _ in yake_keywords], i)
    yake_tp += count_word_overlap_matches([kw for kw, _ in yake_keywords], keywords_list[i])

    # ===== TF-IDF =====
    tfidf_tp += count_matching_keywords(tfidf_keywords[i], i)

    # ===== TextRank =====
    textrank_res = textrank(current_article); extracted_keywords.append([(kw.rank, kw.text) for kw in textrank._.phrases[:]])
    textr_tp += count_matching_keywords([textrank_res._.phrases[j].text for j in range(T)], i)
    textr_tp += count_word_overlap_matches([textrank_res._.phrases[j].text for j in range(T)], keywords_list[i]) """

    # ===== PositionRank =====
    positionrank_res = positionrank(current_article); extracted_keywords.append([(kw.rank, kw.text) for kw in positionrank_res._.phrases[:]])
    #posr_tp += count_matching_keywords([positionrank_res._.phrases[j].text for j in range(T)], i)
    #posr_tp += count_word_overlap_matches([positionrank_res._.phrases[j].text for j in range(T)], keywords_list[i])

    """ # ===== TopicRank =====
    topicrank_res = topicrank(current_article); extracted_keywords.append([(kw.rank, kw.text) for kw in topicrank_res._.phrases[:]])
    topicr_tp += count_matching_keywords([topicrank_res._.phrases[j].text for j in range(T)], i)
    topicr_tp += count_word_overlap_matches([topicrank_res._.phrases[j].text for j in range(T)], keywords_list[i]) """
   


In [ ]:
#textrank.remove_pipe("textrank")
positionrank.remove_pipe("positionrank")
#topicrank.remove_pipe("topicrank")

##### Isolated experiment for generating centroid cluster keywords and creating LLM input

In [23]:
extracted_keywords[0]

[(0.13709422646083025, 'emergency rescue drone products'),
 (0.12996290697413138, 'emergency rescue drones'),
 (0.12545686544883167, 'drone shape design'),
 (0.12214481387754131, 'emergency rescue product design'),
 (0.11487031533872813, 'Drones'),
 (0.11487031533872813, 'drones'),
 (0.1136347103985716, 'Drone modelling'),
 (0.1136347103985716, 'drone modelling'),
 (0.11149970065486095, 'product design'),
 (0.10915165398541238, 'drone kill'),
 (0.10812788178032873, 'product styling design'),
 (0.10474396007626695, 'New Practical Design Approach'),
 (0.0966270426042284, 'the emergency rescue drone product styling design'),
 (0.09529394164561607, 'design elements'),
 (0.0920371643330732, 'user requirements'),
 (0.09151116483047528, 'key user requirements'),
 (0.09104473579853642, 'emergency rescue'),
 (0.08970286548978917, 'user satisfaction'),
 (0.08945007442020597, 'the emergency rescue drone design scheme'),
 (0.08745075560171818, 'the emergency rescue drone product'),
 (0.08745075560

In [24]:
len(extracted_keywords[0])

406

In [25]:
centroid_keywords = sorted(set(cluster_keywords([kw for _, kw in extracted_keywords[0]], 
                                                [score for score, _ in extracted_keywords[0]])), key=lambda x: x[1], reverse=True)

In [31]:
for kw in centroid_keywords[:13]:
    print(kw)

('emergency rescue drone products', 0.13709422646083025)
('Drones', 0.11487031533872813)
('user satisfaction', 0.08970286548978917)
('product development', 0.08390770374231413)
('A New Practical Design Approach', 0.0668149103711736)
('National Emergency Response System', 0.06542972240773436)
('China’s drones', 0.05911219542637088)
('structural elements', 0.058172323885289264)
('scientific and quantitative design methods', 0.051323727648739575)
('Kano', 0.05025778956286968)
('key technologies', 0.04986069398317139)
('the user requirements', 0.04881007726582254)
('FBS', 0.04868337879902244)


In [27]:
len(centroid_keywords)

160

In [28]:
[kw[1] for kw in centroid_keywords]

[0.13709422646083025,
 0.11487031533872813,
 0.08970286548978917,
 0.08390770374231413,
 0.0668149103711736,
 0.06542972240773436,
 0.05911219542637088,
 0.058172323885289264,
 0.051323727648739575,
 0.05025778956286968,
 0.04986069398317139,
 0.04881007726582254,
 0.04868337879902244,
 0.048257006605157964,
 0.04728125762375424,
 0.044137767335420526,
 0.04260006439889978,
 0.03938758249396878,
 0.03902755355616447,
 0.03873961644481122,
 0.03838903943418874,
 0.03779423134946029,
 0.03742147781287709,
 0.036847342026038475,
 0.03670529705043584,
 0.03538638200156055,
 0.03440584798447187,
 0.03325015319732217,
 0.032126214195008725,
 0.028561049445061153,
 0.02799780871765132,
 0.027960995091073054,
 0.02788740198429805,
 0.02761678559400395,
 0.02638575063233146,
 0.02625077082209068,
 0.025772287218584634,
 0.02443338263631088,
 0.023640858263053056,
 0.02343786072627519,
 0.02329485125793785,
 0.02274111315879407,
 0.022501547285701383,
 0.02172411566798305,
 0.021682069347444526,

In [29]:
top_indices = find_knee_with_kneedle([kw[1] for kw in centroid_keywords])

In [30]:
top_indices

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

In [32]:
if len(top_indices) < T:
    if T == 5:
        # Fill up to exactly 5
        extra = list(range(len(top_indices), min(len(centroid_keywords), 5)))
        top_indices.extend(extra)
    elif T == 10:
        # Fill up to at least 8 (if available)
        extra = list(range(len(top_indices), min(len(centroid_keywords), 8)))
        top_indices.extend(extra)

In [ ]:
top_centroid_keywords = " ".join(
    f"{centroid_keywords[idx][0]} ||" for idx in top_indices
)

In [ ]:
top_centroid_keywords

##### Global experiment for generating centroid cluster keywords and creating LLM input

In [ ]:
LLM_input = generate_LLM_input(extracted_keywords)

In [ ]:
with open(f'MDPI_Results/Input/PosR-LLM_Input.txt', 'w') as file:
    for input in LLM_input:
        file.write(f'{input}\n')

***LLM keywords evaluation***

In [ ]:
LLM_keywords = []
with open(f'MDPI_Results/Output/PosR-LLM_Res.txt', 'r') as file:
    for line in file:
        LLM_keywords.append(line.strip().split(', '))

Evaluate for partial, exact, and embedding-based similarity between the LLM keywords and the actual (true) keywords.

In [ ]:
LLM_tp = 0
for i in range(len(LLM_keywords)): LLM_tp += count_word_overlap_matches(LLM_keywords[i], keywords_list[i])
evaluate_method("RAKE LLM", LLM_tp, final_extr_keyword_num, true_keyword_num)

LLM_tp = 0
for i in range(len(LLM_keywords)): LLM_tp += compare_keyword_lists(LLM_keywords[i], keywords_list[i])
evaluate_method("RAKE LLM", LLM_tp, final_extr_keyword_num, true_keyword_num)

""" LLM_tp = 0
for i in range(len(LLM_keywords)): LLM_tp += count_matching_keywords(LLM_keywords[i], i)
evaluate_method("RAKE LLM", LLM_tp, final_extr_keyword_num, true_keyword_num) """

In [ ]:
# Evaluate each method
evaluate_method("RAKE", rake_tp, final_extr_keyword_num, true_keyword_num)
""" evaluate_method("YAKE", yake_tp, final_extr_keyword_num, true_keyword_num)
evaluate_method("TF-IDF", tfidf_tp, final_extr_keyword_num, true_keyword_num)
evaluate_method("TextRank", textr_tp, final_extr_keyword_num, true_keyword_num)
evaluate_method("PositionRank", posr_tp, final_extr_keyword_num, true_keyword_num)
evaluate_method("TopicRank", topicr_tp, final_extr_keyword_num, true_keyword_num) """

#### ***Multiple Executions***
Statistics included

##### No LLM

In [32]:
KE_methods = ['TFIDF', 
              'RAKE',
              'YAKE', 
              'PosR', 
              'TextR', 
              'TopicR']
              #'MPR',
              #'SingleR',
              #'TPageR']

for epoch, ke_method in enumerate(KE_methods): # For testing use ['RAKE']. Otherwise use KE_methods
    if ke_method in ['PosR', 'TextR', 'TopicR']:
        graph_method = spacy.load(spacy_package)
        graph_method.add_pipe("positionrank" if ke_method == 'PosR' else "textrank" if ke_method == 'TextR' else "topicrank")
    
    for t in [5, 10]:
        
        if ke_method == 'TFIDF':
            start_time = time.perf_counter()
            keywords = extract_keywords_tfidf(preprocessed_content, t)
            end_time = time.perf_counter()

        elif ke_method == 'MPR':
            keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                mprank = pke.unsupervised.MultipartiteRank()
                mprank.load_document(input = titles[c] + '. ' + articles[c], stoplist = stoplist, language = 'en', spacy_model = spaCy)
                mprank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
                mprank.candidate_weighting(alpha = 1.1, threshold = 0.74, method = 'average')
                keywords.append([key for key, _ in mprank.get_n_best(n=t)])
            end_time = time.perf_counter()

        elif ke_method == 'SingleR':
            keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                singlerank = pke.unsupervised.SingleRank()
                singlerank.load_document(input = titles[c] + '. ' + articles[c], language = 'en', normalization = None, spacy_model = spaCy)
                singlerank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
                singlerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'})
                keywords.append([key for key, _ in singlerank.get_n_best(n=t)])
            end_time = time.perf_counter()

        elif ke_method == 'TPageR':
            keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                topicalpagerank = pke.unsupervised.TopicalPageRank()
                topicalpagerank.load_document(input = titles[c] + '. ' + articles[c], language = 'en', normalization = None, spacy_model = spaCy)
                topicalpagerank.candidate_selection(grammar = "NP: {<ADJ>*<NOUN|PROPN>+}")
                topicalpagerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'}, lda_model = lda_model_file)
                keywords.append([key for key, _ in topicalpagerank.get_n_best(n=t)])
            end_time = time.perf_counter()

        elif ke_method == 'RAKE':
            keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                rake_extractor = Rake(ranking_metric=Metric.DEGREE_TO_FREQUENCY_RATIO)
                rake_extractor.extract_keywords_from_text(titles[c] + '. ' + articles[c])
                rake_keywords = sorted(set(rake_extractor.get_ranked_phrases_with_scores()), key=lambda x: x[0], reverse=True)[:t]
                keywords.append([kw for _, kw in rake_keywords])
            end_time = time.perf_counter()

        elif ke_method == 'YAKE':
            yake_extractor = yake.KeywordExtractor(lan='en', n=3, dedupLim=0.9, dedupFunc='seqm', windowsSize=1, top=t)
            start_time = time.perf_counter()
            keywords = [[kw for kw, _ in yake_extractor.extract_keywords(titles[c] + '. ' + articles[c])] for c in range(len(articles))]
            end_time = time.perf_counter()
            del yake_extractor
            
        else:
            start_time = time.perf_counter()
            keywords = [[kw.text for kw in graph_method(titles[c] + '. ' + articles[c])._.phrases[:t]] for c in range(len(articles))]
            end_time = time.perf_counter()
        
        #print(f'{ke_method} {t} Time: {(end_time - start_time) / 60.0}')
        total_word_count, total_non_word_count, avg_word_count = process_keywords(keywords)
        #print(f'Average number of total extracted keywords: {extracted_keywords[0] / len(articles)}')
        #print(f'% of Non-Words for {ke_method}: {total_non_word_count / total_word_count}')
        #print(f'Avg word count: {avg_word_count}\n')

        final_extr_keyword_num = 0; true_keyword_num = 0; tp_exact = 0; tp_partial = 0; tp_emb = 0

        for c in range(len(articles)):
            true_keyword_num += len(keywords_list[c])
            final_extr_keyword_num += len(keywords[c])
            
            tp_partial += count_word_overlap_matches(keywords[c], keywords_list[c])
            tp_exact += compare_keyword_lists(keywords[c], keywords_list[c])
            tp_emb += count_matching_keywords(keywords[c], c)

        precision, recall, f1 = [None] * 3, [None] * 3, [None] * 3
        
        precision[0], recall[0], f1[0] = evaluate_method(tp_partial, final_extr_keyword_num, true_keyword_num)
        precision[1], recall[1], f1[1] = evaluate_method(tp_exact, final_extr_keyword_num, true_keyword_num)
        precision[2], recall[2], f1[2] = evaluate_method(tp_emb, final_extr_keyword_num, true_keyword_num)

        # Avg Keys: {final_extr_keyword_num / len(articles)}
        print(f'{ke_method} T={t} Time: {(end_time - start_time) / 60.0} Pre: {precision} Rec: {recall} F1: {f1} Avg WC: {avg_word_count} % NonW: {total_non_word_count / total_word_count}')
        
        gc.collect()
    
    if ke_method in ['PosR', 'TextR', 'TopicR']:
        graph_method.remove_pipe("positionrank" if ke_method == 'PosR' else "textrank" if ke_method == 'TextR' else "topicrank")
        del graph_method
        gc.collect()

gc.collect()


TFIDF T=5 Time: 0.06966015473332542 Pre: [0.35487701441899916, 0.07854113655640373, 0.15453774385072094] Rec: [0.32682393376034996, 0.07233244805499141, 0.1423215122637088] F1: [0.3402732595966168, 0.07530904359141183, 0.14817826935588807] Avg WC: 4.8710771840542835 % NonW: 0.026467003308375415
TFIDF T=10 Time: 0.06610934376667502 Pre: [0.25750636132315524, 0.055640373197625104, 0.11297709923664122] Rec: [0.4743008904858616, 0.10248398687705046, 0.20809248554913296] F1: [0.3337914353251608, 0.07212357759331538, 0.14644604474740255] Avg WC: 9.763358778625955 % NonW: 0.024237685691946835
RAKE T=5 Time: 0.07894679500001682 Pre: [0.11467345207803223, 0.00016963528413910093, 0.024597116200169637] Rec: [0.10560849867208248, 0.0001562255897516013, 0.02265271051398219] F1: [0.10995445673389721, 0.0001626545217957059, 0.023584905660377357] Avg WC: 29.525021204410518 % NonW: 0.09523125538638322
RAKE T=10 Time: 0.07963385829998515 Pre: [0.10322307039864292, 0.0006785411365564037, 0.02790500424088

/home/georgematlis/AUTh/NLP/Keyword-Extraction/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


PosR T=5 Time: 3.99574325855001 Pre: [0.3709923664122137, 0.10415606446140797, 0.1436810856658185] Rec: [0.3416653647867521, 0.0959225121074832, 0.13232307451960632] F1: [0.3557254391672089, 0.09986987638256344, 0.13776837996096292] Avg WC: 13.512298558100085 % NonW: 0.014688343481262946
PosR T=10 Time: 3.8523817534666707 Pre: [0.24189991518235793, 0.07930449533502969, 0.1111111111111111] Rec: [0.44555538197156697, 0.14607092641774722, 0.2046555225745977] F1: [0.3135616513660601, 0.10279808696608213, 0.14402726623055356] Avg WC: 26.167090754877016 % NonW: 0.013354510388642183


/home/georgematlis/AUTh/NLP/Keyword-Extraction/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


TextR T=5 Time: 3.805934224750005 Pre: [0.3190839694656489, 0.07514843087362172, 0.1645462256149279] Rec: [0.2938603343227621, 0.06920793625995939, 0.15153882205905328] F1: [0.30595315549772284, 0.07205595315549773, 0.15777488614183474] Avg WC: 12.888040712468193 % NonW: 0.010924646265218822
TextR T=10 Time: 3.7116570108500127 Pre: [0.20882103477523326, 0.06030534351145038, 0.11823579304495335] Rec: [0.3846274019684424, 0.11107639431338853, 0.21777847211373222] F1: [0.27068330493100984, 0.07817052388543785, 0.15326260238579517] Avg WC: 25.289228159457167 % NonW: 0.0096927823987121


/home/georgematlis/AUTh/NLP/Keyword-Extraction/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


TopicR T=5 Time: 8.192424729250009 Pre: [0.35335029686174724, 0.09923664122137404, 0.18744698897370654] Rec: [0.32541790345258553, 0.09139197000468677, 0.17262927667551944] F1: [0.33880936890045543, 0.09515289525048796, 0.17973324658425502] Avg WC: 9.305343511450381 % NonW: 0.03326952875763376
TopicR T=10 Time: 8.135362063833327 Pre: [0.25750636132315524, 0.07463952502120441, 0.13723494486853266] Rec: [0.4743008904858616, 0.13747851898140914, 0.2527730042180909] F1: [0.3337914353251608, 0.09675114067395964, 0.17789016546643946] Avg WC: 18.367260390161153 % NonW: 0.03172477487878088


0

##### LLM 

In [ ]:
KE_methods = ['TFIDF', 
              'RAKE',
              'YAKE', 
              'PosR', 
              'TextR', 
              'TopicR']
              #'MPR',
              #'SingleR',
              #'TPageR']

additional_keywords = 15 # This variable is used for text extraction methods that output a predefined number of keywords. For such methods, we extract T + additional_keywords keywords to give the LLM more options to choose from.

for epoch, ke_method in enumerate(KE_methods): # For testing use ['RAKE']. Otherwise use KE_methods
    print(ke_method)
    if ke_method in ['PosR', 'TextR', 'TopicR']:
        graph_method = spacy.load(spacy_package)
        graph_method.add_pipe("positionrank" if ke_method == 'PosR' else "textrank" if ke_method == 'TextR' else "topicrank")
    
    for t in [t + additional_keywords for t in [5, 10]]:
        
        if ke_method == 'TFIDF':
            start_time = time.perf_counter()
            extracted_keywords = extract_keywords_tfidf(preprocessed_content, t)

        elif ke_method == 'MPR':
            extracted_keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                mprank = pke.unsupervised.MultipartiteRank()
                mprank.load_document(input = titles[c] + '. ' + articles[c], stoplist = stoplist, language = 'en', spacy_model = spaCy)
                mprank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
                mprank.candidate_weighting(alpha = 1.1, threshold = 0.74, method = 'average')
                extracted_keywords.append([(score, key) for key, score in mprank.get_all_candidates()])

        elif ke_method == 'SingleR':
            extracted_keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                singlerank = pke.unsupervised.SingleRank()
                singlerank.load_document(input = titles[c] + '. ' + articles[c], language = 'en', normalization = None, spacy_model = spaCy)
                singlerank.candidate_selection(pos = {'NOUN', 'PROPN', 'ADJ'})
                singlerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'})
                extracted_keywords.append([(score, key) for key, score in singlerank.get_all_candidates()])

        elif ke_method == 'TPageR':
            extracted_keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                topicalpagerank = pke.unsupervised.TopicalPageRank()
                topicalpagerank.load_document(input = titles[c] + '. ' + articles[c], language = 'en', normalization = None, spacy_model = spaCy)
                topicalpagerank.candidate_selection(grammar = "NP: {<ADJ>*<NOUN|PROPN>+}")
                topicalpagerank.candidate_weighting(window = 10, pos = {'NOUN', 'PROPN', 'ADJ'}, lda_model = lda_model_file)
                extracted_keywords.append([(score, key) for key, score in topicalpagerank.get_all_candidates()])
            

        elif ke_method == 'RAKE':
            extracted_keywords = []
            start_time = time.perf_counter()
            for c in range(len(articles)):
                rake_extractor = Rake(ranking_metric=Metric.DEGREE_TO_FREQUENCY_RATIO)
                rake_extractor.extract_keywords_from_text(titles[c] + '. ' + articles[c])
                rake_keywords = sorted(set(rake_extractor.get_ranked_phrases_with_scores()), key=lambda x: x[0], reverse=True)
                extracted_keywords.append([(score, kw) for score, kw in rake_keywords])

        elif ke_method == 'YAKE':
            yake_extractor = yake.KeywordExtractor(lan='en', n=3, dedupLim=0.9, dedupFunc='seqm', windowsSize=1, top=t)
            start_time = time.perf_counter()
            extracted_keywords = [[(score, kw) for kw, score in yake_extractor.extract_keywords(titles[c] + '. ' + articles[c])] for c in range(len(articles))]
            del yake_extractor
            
        else:
            start_time = time.perf_counter()
            extracted_keywords = [[(kw.rank, kw.text) for kw in graph_method(titles[c] + '. ' + articles[c])._.phrases] for c in range(len(articles))]
        



        # LLM_input = generate_LLM_input(extracted_keywords) # Use this to evaluate LLM performance
        if ke_method != 'TFIDF' or ke_method != 'TopicR':
            extracted_keywords = get_top_centroids(extracted_keywords) # Use this to evaluate non-LLM performance (just using the top centroids). The top centroids are basically some keywords that are distinct from each other
            end_time = time.perf_counter()


            # OPTIONAL: Run the below code to evaluate the quality of the top centroids
            final_extr_keyword_num = 0; true_keyword_num = 0; tp_exact = 0; tp_partial = 0; tp_emb = 0

            for c in range(len(articles)):
                true_keyword_num += len(keywords_list[c])
                final_extr_keyword_num += len(extracted_keywords[c])
                
                tp_partial += count_word_overlap_matches(extracted_keywords[c], keywords_list[c])
                tp_exact += compare_keyword_lists(extracted_keywords[c], keywords_list[c])
                tp_emb += count_matching_keywords(extracted_keywords[c], c)

            precision, recall, f1 = [None] * 3, [None] * 3, [None] * 3
            
            precision[0], recall[0], f1[0] = evaluate_method(tp_partial, final_extr_keyword_num, true_keyword_num)
            precision[1], recall[1], f1[1] = evaluate_method(tp_exact, final_extr_keyword_num, true_keyword_num)
            precision[2], recall[2], f1[2] = evaluate_method(tp_emb, final_extr_keyword_num, true_keyword_num)

            print(f'{ke_method} T={t} Time: {(end_time - start_time) / 60.0} Pre: {precision} Rec: {recall} F1: {f1}')
            

            with open(f'MDPI_Results/LLM/Input/{ke_method}{t}-LLM_Input.txt', 'w') as file:
                for input in LLM_input:
                    file.write(f'{input}\n')

        
        gc.collect()
    
    if ke_method in ['PosR', 'TextR', 'TopicR']:
        graph_method.remove_pipe("positionrank" if ke_method == 'PosR' else "textrank" if ke_method == 'TextR' else "topicrank")
        del graph_method
        gc.collect()

gc.collect()


#### Create word count boxplots

In [ ]:
analyze_word_counts([
    f'{parent_path}{target_folder}Word_Counts/TFIDF5.txt',
    f'{parent_path}{target_folder}Word_Counts/RAKE5.txt',
    f'{parent_path}{target_folder}Word_Counts/YAKE5.txt',
    f'{parent_path}{target_folder}Word_Counts/TextR5.txt',
    f'{parent_path}{target_folder}Word_Counts/PositionR5.txt',
    f'{parent_path}{target_folder}Word_Counts/TopicR5.txt'
])